# Chronoscopes — turbopuffer Ingestion Notebook

Embeds 4.47M historical newspaper chunks with `all-MiniLM-L6-v2` on T4 GPU  
and upserts them to turbopuffer namespace `chronoscopes-v2` with BM25 + ANN schema.

**Before running:**
1. Set Runtime → Change runtime type → **GPU T4**
2. Add `TURBOPUFFER_API_KEY` to Kaggle Secrets (Add-ons → Secrets)
3. Attach the `chronoscopes-wwii-parquet` dataset (Input → Add dataset)

**Estimated time:** ~3.5 hours (dominated by turbopuffer WAL rate limit)  
**Resumable:** checkpoint saved after each shard — restart cell 5 to continue after a crash.

In [11]:
# Cell 1 — Install dependencies
!pip install turbopuffer sentence-transformers pyarrow tqdm --quiet

In [12]:
# Cell 2 — Imports and config
import os, glob, time, json
import numpy as np
import pyarrow.parquet as pq
import turbopuffer as tpuf
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import torch

# ── Config ────────────────────────────────────────────────────────────────────
# TURBOPUFFER_API_KEY = os.environ.get('TURBOPUFFER_API_KEY') or \
#     open('/root/.kaggle/secrets/TURBOPUFFER_API_KEY').read().strip()

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
TURBOPUFFER_API_KEY = user_secrets.get_secret("TURBOPUFFER_API_KEY")


NAMESPACE      = 'chronoscopes-v2'
PARQUET_DIR    = '/kaggle/input/datasets/sharmilaraghu/chronoscopes-wwii-parquet'
CHECKPOINT_FILE = '/kaggle/working/checkpoint.json'
ENCODE_BATCH   = 512    # T4 16GB VRAM — safe with all-MiniLM-L6-v2
UPSERT_BATCH   = 500    # vectors per turbopuffer write call
UPSERT_DELAY   = 1.05   # seconds between writes (WAL limit: 1/sec)

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:             {torch.cuda.get_device_name(0)}')
    print(f'VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch version: 2.10.0+cu128
CUDA available:  True
GPU:             Tesla T4
VRAM:            15.6 GB


In [13]:
# Cell 3 — Load embedding model
print('Loading sentence-transformers/all-MiniLM-L6-v2 ...')
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
model.max_seq_length = 256    # model hard limit: 256 tokens
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f'Model loaded on: {device}')

# Sanity-check: embed one string, confirm 384 dimensions
test_vec = model.encode(['test'], normalize_embeddings=True)[0]
assert len(test_vec) == 384, f'Expected 384-dim, got {len(test_vec)}'
print(f'Embedding dim: {len(test_vec)} ✓')

Loading sentence-transformers/all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on: cuda
Embedding dim: 384 ✓


In [14]:
# Cell 4 — Init turbopuffer client + namespace
client = tpuf.Client(api_key=TURBOPUFFER_API_KEY, region='aws-us-east-1')
ns = client.namespace(NAMESPACE)
print(f'Namespace: {NAMESPACE}')
exists = ns.exists()
print(f'Already exists: {exists}')
if exists:
    meta = ns.metadata()
    print(f'Current vector count: {meta.approx_count:,}')

Namespace: chronoscopes-v2
Already exists: False


In [15]:
# Cell 5 — Checkpoint helpers

def load_checkpoint() -> dict:
    """Returns {'last_shard': int, 'last_batch_start': int} or defaults."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            return json.load(f)
    return {'last_shard': -1, 'last_batch_start': -1}

def save_checkpoint(shard_idx: int, batch_start: int = -1) -> None:
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({'last_shard': shard_idx, 'last_batch_start': batch_start}, f)

checkpoint = load_checkpoint()
print(f'Resuming from: shard {checkpoint["last_shard"] + 1}')

Resuming from: shard 0


In [ ]:
# Cell 6 — turbopuffer schema (BM25 + ANN)
# Passed only on the FIRST write to establish the namespace schema.
# Subsequent writes omit it — turbopuffer infers from the first write.

TPUF_SCHEMA = {
      # BM25 enabled on free-text fields — correct key is 'full_text_search'
      'text':           {'type': 'string', 'full_text_search': True},
      'headline':       {'type': 'string', 'full_text_search': True},

      # Filterable metadata
      'newspaper_name': {'type': 'string'},
      'date':           {'type': 'string'},
      'year':           {'type': 'uint'},
      'city':           {'type': 'string'},
      'state':          {'type': 'string'},
      'era':            {'type': 'string'},

      # Stored as strings (not filterable, but accepted by turbopuffer)
      'latitude':       {'type': 'string'},
      'longitude':      {'type': 'string'},
      'chunk_idx':      {'type': 'uint'},
      'total_chunks':   {'type': 'uint'},
      'article_id':     {'type': 'string'},
  }

In [ ]:
# Cell 7 — Main ingestion loop (resumable)

shard_paths = sorted(glob.glob(os.path.join(PARQUET_DIR, 'chronoscopes_wwii_shard_*.parquet')))
print(f'Found {len(shard_paths)} shards')

checkpoint      = load_checkpoint()
last_shard      = checkpoint['last_shard']
is_first_write  = not ns.exists()   # only pass schema on very first write

total_upserted  = 0
job_start       = time.time()

for shard_idx, shard_path in enumerate(shard_paths):

    if shard_idx <= last_shard:
        print(f'[{shard_idx}/{len(shard_paths)-1}] Skipping (already done)')
        continue

    shard_name = os.path.basename(shard_path)
    print(f'\n=== Shard {shard_idx}/{len(shard_paths)-1}: {shard_name} ===')

    # ── Load shard ────────────────────────────────────────────────────────────
    df = pq.read_table(shard_path).to_pandas()
    print(f'  Loaded {len(df):,} rows')

    # ── Build embed strings: "headline: text" or just "text" ──────────────────
    embed_strings = [
        f"{row.headline}: {row.text}".strip() if row.headline else str(row.text)
        for row in df.itertuples(index=False)
    ]

    # ── Encode on GPU ─────────────────────────────────────────────────────────
    print(f'  Encoding {len(embed_strings):,} chunks on {device} ...')
    t_enc = time.time()
    vectors = model.encode(
        embed_strings,
        batch_size=ENCODE_BATCH,
        normalize_embeddings=True,   # required for cosine_distance
        show_progress_bar=True,
        device=device,
    )
    enc_secs = time.time() - t_enc
    print(f'  Encoded in {enc_secs:.1f}s  ({len(embed_strings)/enc_secs:,.0f} docs/sec)')

    # ── Build turbopuffer rows ─────────────────────────────────────────────────
    rows = []
    for row, vec in zip(df.itertuples(index=False), vectors):
        rows.append({
            'id':             str(row.doc_id)[:64],
            'vector':         vec.tolist(),
            'text':           str(row.text or '')[:1000],
            'headline':       str(row.headline or '')[:200],
            'newspaper_name': str(row.newspaper_name or '')[:200],
            'date':           str(row.date or '')[:10],
            'year':           int(row.year) if row.year else 0,
            'city':           str(row.city or '')[:100],
            'state':          str(row.state or '')[:10],
            'era':            'WWII',
            'latitude':       str(row.latitude) if row.latitude is not None else '',
            'longitude':      str(row.longitude) if row.longitude is not None else '',
            'chunk_idx':      int(row.chunk_idx) if row.chunk_idx is not None else 0,
            'total_chunks':   int(row.total_chunks) if row.total_chunks is not None else 1,
            'article_id':     str(row.article_id or '')[:64],
        })

    # ── Upsert to turbopuffer in batches ──────────────────────────────────────
    print(f'  Upserting {len(rows):,} rows (batch={UPSERT_BATCH}) ...')
    t_upsert = time.time()
    batches = range(0, len(rows), UPSERT_BATCH)

    for batch_start in tqdm(batches, desc='  upserting'):
        batch = rows[batch_start: batch_start + UPSERT_BATCH]

        # Pass schema only on the very first write to the namespace
        kwargs = {'upsert_rows': batch, 'distance_metric': 'cosine_distance'}
        if is_first_write and batch_start == 0:
            kwargs['schema'] = TPUF_SCHEMA
            is_first_write = False

        ns.write(**kwargs)
        time.sleep(UPSERT_DELAY)   # honour 1 WAL write/sec

    upsert_secs = time.time() - t_upsert
    total_upserted += len(rows)
    print(f'  Upserted in {upsert_secs:.1f}s  ({len(rows)/upsert_secs:,.0f} docs/sec)')

    save_checkpoint(shard_idx)
    elapsed_total = time.time() - job_start
    shards_done   = shard_idx - last_shard
    shards_left   = len(shard_paths) - shard_idx - 1
    eta_secs      = (elapsed_total / shards_done) * shards_left if shards_done else 0
    print(f'  ✓ Checkpoint saved  |  ETA for remaining {shards_left} shards: {eta_secs/3600:.1f} h')

print(f'\n=== INGESTION COMPLETE ===')
print(f'  Namespace:        {NAMESPACE}')
print(f'  Total upserted:   {total_upserted:,}')
print(f'  Wall time:        {(time.time()-job_start)/3600:.2f} h')

In [ ]:
# Cell 8 — Verification
print('=== POST-INGEST VERIFICATION ===')

# 1. Vector count
meta = ns.metadata()
print(f'approx_count: {meta.approx_count:,}  (expected ~4,472,113)')

# 2. ANN semantic query
q = 'air raid sirens bombing night city'
q_vec = model.encode([q], normalize_embeddings=True)[0].tolist()
results = ns.query(
    rank_by=('vector', 'ANN', q_vec),
    top_k=3,
    filters=('city', 'Eq', 'Washington'),
    include_attributes=['text', 'headline', 'year', 'city'],
)
print(f'\nANN results for: "{q}" filtered to Washington')
for r in results:
    print(f'  [{r.attributes["year"]}] {r.attributes["headline"][:60]}')
    print(f'  {r.attributes["text"][:120]}...')

# 3. BM25 keyword query
results_bm25 = ns.query(
    rank_by=('bm25', 'text', 'Pearl Harbor attack'),
    top_k=3,
    include_attributes=['text', 'headline', 'year', 'city'],
)
print(f'\nBM25 results for: "Pearl Harbor attack"')
for r in results_bm25:
    print(f'  [{r.attributes["year"]}] {r.attributes["headline"][:60]}')

# 4. Hybrid RRF
results_rrf = ns.query(
    rank_by=('rrf', [
        ('vector', 'ANN', q_vec),
        ('bm25',   'text', q),
    ]),
    top_k=3,
    include_attributes=['text', 'headline', 'year', 'city'],
)
print(f'\nHybrid RRF results for: "{q}"')
for r in results_rrf:
    print(f'  [{r.attributes["year"]}] {r.attributes["city"]} — {r.attributes["headline"][:60]}')

# 5. Year filter sanity check
results_year = ns.query(
    rank_by=('vector', 'ANN', q_vec),
    top_k=5,
    filters=('year', 'Eq', 1944),
    include_attributes=['year'],
)
years = [r.attributes['year'] for r in results_year]
print(f'\nYear filter (year=1944): got years = {years}  ← all should be 1944')
assert all(y == 1944 for y in years), 'Year filter broken!'
print('All checks passed ✓')